In [1]:
print(1)

1


In [2]:
import numpy as np
from datetime import datetime

# dataset min & max (hardcode from your stats)
IRR_MIN = 0.9022
IRR_MAX = 7.3234

def get_model_irradiance(data):
    """
    Convert OpenWeather API → dataset-scale irradiance
    """

    clouds = data["clouds"]["all"]
    current_time = data["dt"]
    sunrise = data["sys"]["sunrise"]
    sunset = data["sys"]["sunset"]

    current = datetime.fromtimestamp(current_time)
    rise = datetime.fromtimestamp(sunrise)
    set_ = datetime.fromtimestamp(sunset)

    # Night
    if current < rise or current > set_:
        return 0.0

    # Daylight
    daylight = (set_ - rise).total_seconds()
    elapsed = (current - rise).total_seconds()

    solar_factor = np.sin(np.pi * elapsed / daylight)
    cloud_factor = (100 - clouds) / 100

    # normalized (0–1)
    norm_irr = solar_factor * cloud_factor

    # 🔥 SCALE TO DATASET RANGE
    scaled_irr = IRR_MIN + norm_irr * (IRR_MAX - IRR_MIN)

    return round(max(0, scaled_irr), 4)

In [4]:
import requests
import os
import dotenv

dotenv.load_dotenv()

response = requests.get("https://api.openweathermap.org/data/2.5/weather", params={
    "q": "chennai,in",
    "appid": os.getenv("OPENWEATHER_API_KEY")
})
data = response.json()
# irr = get_model_irradiance(data)
# print("☀️ Irradiance:", irr, "W/m²")
print(data)

{'coord': {'lon': 80.2785, 'lat': 13.0878}, 'weather': [{'id': 801, 'main': 'Clouds', 'description': 'few clouds', 'icon': '02d'}], 'base': 'stations', 'main': {'temp': 308.44, 'feels_like': 315.44, 'temp_min': 307.64, 'temp_max': 308.69, 'pressure': 1009, 'humidity': 58, 'sea_level': 1009, 'grnd_level': 1009}, 'visibility': 6000, 'wind': {'speed': 5.66, 'deg': 200}, 'clouds': {'all': 20}, 'dt': 1776579034, 'sys': {'type': 2, 'id': 2104103, 'country': 'IN', 'sunrise': 1776558238, 'sunset': 1776603107}, 'timezone': 19800, 'id': 1264527, 'name': 'Chennai', 'cod': 200}
